# NBA Stats Database
Fetches 2025-26 season stats from NBA.com and saves to a local SQLite database for SQL querying.

## 1. Install Dependencies

In [ ]:
from nba_api.stats.endpoints import leaguedashplayerstats, leaguedashteamstats
import pandas as pd
import sqlite3
from nba_api.stats.endpoints import leaguegamelog
import time
import os
import math

## 2. Fetch Player Stats from NBA.com

In [ ]:

SEASON = '2025-26'

# --- Player Stats ---
print('Fetching player stats...')
player_stats = leaguedashplayerstats.LeagueDashPlayerStats(season=SEASON)
df_players = pd.DataFrame(player_stats.get_data_frames()[0])
print(f'Players fetched: {len(df_players)}')
df_players.head()

In [ ]:
# --- Team Stats ---
print('Fetching team stats...')
team_stats = leaguedashteamstats.LeagueDashTeamStats(season=SEASON)
df_teams = pd.DataFrame(team_stats.get_data_frames()[0])
print(f'Teams fetched: {len(df_teams)}')
df_teams.head()

## 3. Save to SQLite Database

In [ ]:
from nba_api.stats.endpoints import playerindex
import time

# --- Fetch player bio ---
print('Fetching player bio...')
time.sleep(1)
bio_raw = playerindex.PlayerIndex(season='2025-26')
df_bio = pd.DataFrame(bio_raw.get_data_frames()[0])


def height_to_inches(h):
    try:
        ft, inch = str(h).split('-')
        return int(ft) * 12 + int(inch)
    except:
        return None


df_bio['PLAYER_NAME'] = df_bio['PLAYER_FIRST_NAME'] + ' ' + df_bio['PLAYER_LAST_NAME']
df_bio['HEIGHT_INCHES'] = df_bio['HEIGHT'].apply(height_to_inches)
df_bio['WEIGHT'] = pd.to_numeric(df_bio['WEIGHT'], errors='coerce')
print(f'Bio fetched: {len(df_bio)} players')
df_bio.head()

In [ ]:
# --- Fetch game logs ---
print('Fetching game logs...')
time.sleep(1)
raw = leaguegamelog.LeagueGameLog(season='2025-26', player_or_team_abbreviation='P')
df_games = pd.DataFrame(raw.get_data_frames()[0])
df_games['HOME_AWAY'] = df_games['MATCHUP'].apply(lambda x: 'Home' if 'vs.' in x else 'Away')
print(f'Game logs fetched: {len(df_games)} rows')

# --- Merge player bio into player stats (single unified table) ---
bio_cols = [c for c in ['PLAYER_NAME', 'HEIGHT', 'HEIGHT_INCHES', 'WEIGHT',
                        'COUNTRY', 'COLLEGE', 'DRAFT_YEAR', 'DRAFT_NUMBER', 'POSITION']
            if c in df_bio.columns]
df_players_merged = df_players.merge(df_bio[bio_cols], on='PLAYER_NAME', how='left')
print(f'Merged player stats + bio: {len(df_players_merged)} players, {len(df_players_merged.columns)} columns')

# --- Save to DB ---
conn = sqlite3.connect('nba_stats.db')

df_players_merged.to_sql('player_stats', conn, if_exists='replace', index=False)
df_teams.to_sql('team_stats', conn, if_exists='replace', index=False)
df_games.to_sql('game_logs', conn, if_exists='replace', index=False)

# Remove stale separate bio table if it exists from old runs
conn.execute('DROP TABLE IF EXISTS player_bio')

# --- Views ---
views = {
    'players_with_team': """
                         SELECT p.PLAYER_NAME,
                                p.TEAM_ABBREVIATION,
                                p.GP,
                                p.PTS,
                                p.REB,
                                p.AST,
                                p.STL,
                                p.BLK,
                                p.FG_PCT,
                                p.FG3_PCT,
                                p.FT_PCT,
                                p.HEIGHT,
                                p.HEIGHT_INCHES,
                                p.WEIGHT,
                                p.POSITION,
                                p.COUNTRY,
                                p.COLLEGE,
                                t.W,
                                t.L,
                                t.W_PCT AS TEAM_WIN_PCT
                         FROM player_stats p
                                  JOIN team_stats t ON p.TEAM_ID = t.TEAM_ID
                         """,
    'games_enriched': """
                      SELECT g.PLAYER_NAME,
                             g.TEAM_ABBREVIATION,
                             g.GAME_DATE,
                             g.MATCHUP,
                             g.HOME_AWAY,
                             g.WL,
                             g.PTS,
                             g.REB,
                             g.AST,
                             g.STL,
                             g.BLK,
                             g.TOV,
                             g.FG_PCT,
                             g.FG3_PCT,
                             g.FT_PCT,
                             g.MIN,
                             p.PTS   AS SEASON_AVG_PTS,
                             p.REB   AS SEASON_AVG_REB,
                             p.HEIGHT_INCHES,
                             p.WEIGHT,
                             p.POSITION,
                             t.W_PCT AS TEAM_WIN_PCT
                      FROM game_logs g
                               LEFT JOIN player_stats p ON g.PLAYER_NAME = p.PLAYER_NAME
                               LEFT JOIN team_stats t ON g.TEAM_ID = t.TEAM_ID
                      """
}

for name, sql in views.items():
    conn.execute(f'DROP VIEW IF EXISTS {name}')
    conn.execute(f'CREATE VIEW {name} AS {sql}')

conn.execute('DROP VIEW IF EXISTS players_full')  # no longer needed; bio is now in player_stats

conn.commit()
conn.close()
print('Database saved as nba_stats.db')

## 4. Database Schema & How to Query

The database (`nba_stats.db`) has **three tables** and **one view**:

| Name | Type | Description |
|------|------|-------------|
| `player_stats` | table | Season averages **+ bio** (HEIGHT, WEIGHT, POSITION, COLLEGE, COUNTRY, DRAFT info) — one row per player |
| `team_stats` | table | Team-level season totals and win/loss record |
| `game_logs` | table | **Every individual game played** — ~26,000 rows, one per player per game |
| `players_with_team` | view | `player_stats` joined with `team_stats` (adds W, L, TEAM_WIN_PCT) |
| `games_enriched` | view | `game_logs` joined with `player_stats` and `team_stats` (adds season averages, bio, team win %) |

---

### Viewing a player's game log (all games they played this season)

Query the `game_logs` table — one row per player per game:

```sql
SELECT GAME_DATE, MATCHUP, HOME_AWAY, WL, MIN, PTS, REB, AST, STL, BLK, FG_PCT
FROM game_logs
WHERE PLAYER_NAME = 'LeBron James'
ORDER BY GAME_DATE DESC
```

For bio context (height, position, etc.) alongside each game, use the enriched view:

```sql
SELECT GAME_DATE, MATCHUP, HOME_AWAY, WL, PTS, REB, AST,
       HEIGHT, WEIGHT, POSITION, SEASON_AVG_PTS
FROM games_enriched
WHERE PLAYER_NAME = 'Stephen Curry'
ORDER BY GAME_DATE DESC
```

---

### Other useful game log queries

Best single-game performances this season:
```sql
SELECT PLAYER_NAME, GAME_DATE, MATCHUP, PTS, REB, AST
FROM game_logs
ORDER BY PTS DESC
LIMIT 20
```

Filter by home/away or win/loss:
```sql
SELECT PLAYER_NAME, ROUND(AVG(PTS), 1) AS AVG_PTS, COUNT(*) AS GAMES
FROM game_logs
WHERE HOME_AWAY = 'Away' AND WL = 'W'
GROUP BY PLAYER_NAME
ORDER BY AVG_PTS DESC
LIMIT 15
```

Players who scored 40+ in a single game:
```sql
SELECT PLAYER_NAME, GAME_DATE, MATCHUP, PTS, REB, AST, MIN
FROM game_logs
WHERE PTS >= 40
ORDER BY PTS DESC
```

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import json

# ── AI Backend Toggle ──────────────────────────────────────────────────────────
# Set USE_CLAUDE = True  to call the Anthropic Claude API (requires the
#   ANTHROPIC_API_KEY environment variable and `pip install anthropic`).
# Set USE_CLAUDE = False to fall back to a local Ollama model (requires Ollama
#   running locally with the model specified in LOCAL_MODEL pulled).
USE_CLAUDE = True

CLAUDE_MODEL = 'claude-sonnet-4-6'  # Best reasoning for complex NBA analysis
LOCAL_MODEL = 'gemma4'  # Ollama model name (used when USE_CLAUDE = False)
model = CLAUDE_MODEL if USE_CLAUDE else LOCAL_MODEL
DB_PATH = "nba_stats.db"

# Ollama is only imported when the local backend is active so the notebook
# works without it installed when USE_CLAUDE = True.
if not USE_CLAUDE:
    import ollama
    from ollama import chat, ChatResponse

In [ ]:
# ── Claude API Helpers ─────────────────────────────────────────────────────────
if USE_CLAUDE:
    import anthropic
    import inspect as _inspect
    import typing as _typing

    _claude_client = anthropic.Anthropic()

    # Max characters for a single tool result stored in message history.
    # ~200K chars ≈ ~50K tokens; keeps accumulated history well under the 1M limit.
    _TOOL_RESULT_CHAR_LIMIT = 200_000

    _PYTHON_TO_JSON_TYPE = {
        str: 'string', int: 'integer', float: 'number',
        bool: 'boolean', list: 'array', dict: 'object',
    }

    def _parse_google_docstring(fn):
        doc = (_inspect.getdoc(fn) or '').strip()
        if not doc:
            return '', {}
        lines = doc.splitlines()
        description_lines, param_descs = [], {}
        in_args, current_param = False, None
        for line in lines:
            stripped = line.strip()
            if stripped.lower() in ('args:', 'arguments:', 'parameters:'):
                in_args = True
                continue
            if stripped.endswith(':') and not line.startswith('    ') and in_args:
                in_args = False
                current_param = None
                continue
            if in_args:
                if line.startswith('    ') and ':' in stripped:
                    param, _, desc = stripped.partition(':')
                    current_param = param.strip()
                    param_descs[current_param] = desc.strip()
                elif current_param and stripped:
                    param_descs[current_param] = (param_descs[current_param] + ' ' + stripped).strip()
            else:
                description_lines.append(stripped)
        return ' '.join(l for l in description_lines if l), param_descs

    def function_to_claude_tool(fn):
        sig = _inspect.signature(fn)
        description, param_descs = _parse_google_docstring(fn)
        if not description:
            description = fn.__name__.replace('_', ' ').capitalize()
        properties, required = {}, []
        for param_name, param in sig.parameters.items():
            ann = param.annotation
            if ann is _inspect.Parameter.empty:
                json_type = 'string'
            else:
                origin = getattr(ann, '__origin__', None)
                if origin is _typing.Union:
                    non_none = [a for a in ann.__args__ if a is not type(None)]
                    inner = non_none[0] if non_none else str
                    json_type = _PYTHON_TO_JSON_TYPE.get(inner, 'string')
                else:
                    json_type = _PYTHON_TO_JSON_TYPE.get(ann, 'string')
            prop = {'type': json_type}
            if param_descs.get(param_name):
                prop['description'] = param_descs[param_name]
            properties[param_name] = prop
            if param.default is _inspect.Parameter.empty:
                required.append(param_name)
        return {
            'name': fn.__name__,
            'description': description,
            'input_schema': {'type': 'object', 'properties': properties, 'required': required},
        }

    def fix_sql_claude(bad_sql, error_message):
        schema = get_schema()
        system = [{
            'type': 'text',
            'text': (
                "You are a SQLite SQL expert helping fix NBA stat queries.\n\n"
                "SQLite alias rules to follow:\n"
                "- Aliases CANNOT start with a digit (3P_Index → Three_P_Index)\n"
                "- Aliases with hyphens MUST be wrapped in backticks (Two-Way → `Two-Way`)\n"
                "- The ORDER BY alias must exactly match the SELECT alias\n"
                "- Use only letters, digits, and underscores in alias names\n\n"
                f"Database schema:\n{schema}"
            ),
            'cache_control': {'type': 'ephemeral'},
        }]
        prompt = (
            f"The following SQL query raised a syntax error.\n"
            f"SQL: {bad_sql}\nError: {error_message}\n\n"
            "Return ONLY the corrected SQL — no explanation, no markdown fences."
        )
        response = _claude_client.messages.create(
            model=CLAUDE_MODEL, max_tokens=2048, system=system,
            messages=[{'role': 'user', 'content': prompt}],
        )
        fixed = response.content[0].text.strip()
        if fixed.startswith('```'):
            fixed = fixed.split('```')[1]
            if fixed.startswith('sql'):
                fixed = fixed[3:]
        return fixed.strip()

    def run_agentic_loop_claude(messages):
        schema = get_schema()
        system = [{
            'type': 'text',
            'text': (
                "You are an expert NBA statistician and data analyst. "
                "You have access to a SQLite database containing the full 2025-26 NBA season: "
                "player stats, team stats, individual game logs, and player bio data.\n\n"
                "Use the available tools to query the database, compute custom statistics, "
                "and visualise results. Always prefer data-driven answers over guesses.\n\n"
                f"Database schema:\n{schema}\n\n"
                "Important SQLite rules:\n"
                "- Aliases cannot start with a digit (use Three_P_Index, not 3P_Index)\n"
                "- Aliases with hyphens must be wrapped in backticks\n"
                "- The ORDER BY alias must exactly match the SELECT alias\n\n"
                "Query size rules:\n"
                "- Always include LIMIT in every SQL query (max 500 rows returned)\n"
                "- For leaderboards, LIMIT 20-50 is usually sufficient\n"
                "- For aggregated stats (GROUP BY), no LIMIT needed if few groups"
            ),
            'cache_control': {'type': 'ephemeral'},
        }]

        total_input_tokens = 0
        total_output_tokens = 0
        total_cache_read_tokens = 0
        total_cache_creation_tokens = 0

        while True:
            python_fns = assemble_tools()
            tool_registry = {fn.__name__: fn for fn in python_fns}
            claude_tools = [function_to_claude_tool(fn) for fn in python_fns]

            response = _claude_client.messages.create(
                model=CLAUDE_MODEL, max_tokens=16000,
                system=system, tools=claude_tools, messages=messages,
            )

            if hasattr(response, 'usage'):
                total_input_tokens += response.usage.input_tokens
                total_output_tokens += response.usage.output_tokens
                total_cache_read_tokens += getattr(response.usage, 'cache_read_input_tokens', 0) or 0
                total_cache_creation_tokens += getattr(response.usage, 'cache_creation_input_tokens', 0) or 0

            if response.stop_reason != 'tool_use':
                text_parts = [b.text for b in response.content if b.type == 'text']
                print(
                    f"\nTokens — input: {total_input_tokens:,} | output: {total_output_tokens:,}"
                    f" | cache read: {total_cache_read_tokens:,} | cache write: {total_cache_creation_tokens:,}"
                )
                return ' '.join(text_parts) or ''

            assistant_content = []
            for block in response.content:
                if block.type == 'text':
                    assistant_content.append({'type': 'text', 'text': block.text})
                elif block.type == 'tool_use':
                    assistant_content.append({
                        'type': 'tool_use', 'id': block.id,
                        'name': block.name, 'input': block.input,
                    })
            messages.append({'role': 'assistant', 'content': assistant_content})

            tool_results = []
            for block in response.content:
                if block.type != 'tool_use':
                    continue
                fn = tool_registry.get(block.name)
                if fn is None:
                    result = {'error': f'Unknown tool: {block.name}'}
                else:
                    try:
                        result = fn(**block.input)
                    except Exception as e:
                        result = {'error': str(e)}

                content = json.dumps(result)
                # Hard cap: truncate oversized results before they enter message history
                if len(content) > _TOOL_RESULT_CHAR_LIMIT:
                    content = content[:_TOOL_RESULT_CHAR_LIMIT] + '... [TRUNCATED: result too large]'
                    print(f"[Warning] Tool result from {block.name} truncated to {_TOOL_RESULT_CHAR_LIMIT:,} chars")

                tool_results.append({
                    'type': 'tool_result', 'tool_use_id': block.id,
                    'content': content,
                })
            messages.append({'role': 'user', 'content': tool_results})


In [ ]:
def get_schema():
    conn = sqlite3.connect(DB_PATH)
    schema = conn.execute(
        "SELECT sql FROM sqlite_master WHERE type='table'"
    ).fetchall()
    conn.close()
    return "\n".join([s[0] for s in schema if s[0]])

In [ ]:
def run_query(sql, retry=True):
    conn = sqlite3.connect(DB_PATH)
    try:
        df = pd.read_sql(sql, conn)
        conn.close()
        return df
    except Exception as e:
        conn.close()
        if retry:
            fixed_sql = fix_sql(sql, str(e))
            print(f"Fixed SQL: {fixed_sql}")
            return run_query(fixed_sql, retry=False)  # retry once with fix
        else:
            print("Fix agent also failed.")
            raise

In [ ]:
import re


def clean_sql(sql):
    # Find AS aliases and wrap ones with hyphens or spaces in backticks
    def fix_alias(match):
        alias = match.group(1)
        if '-' in alias or ' ' in alias:
            return f'AS `{alias}`'
        return f'AS {alias}'

    # Fix AS aliases
    sql = re.sub(r'AS\s+([^\s,\)FROM]+)', fix_alias, sql)

    # Also fix ORDER BY references to the same alias
    sql = re.sub(r'ORDER BY\s+([^\s]+)', lambda m: f'ORDER BY `{m.group(1)}`'
    if '-' in m.group(1) else f'ORDER BY {m.group(1)}', sql)

    return sql

In [ ]:
def fix_sql(bad_sql, error_message):
    """Repair a broken SQLite query.  Dispatches to Claude or Ollama per USE_CLAUDE."""
    print("SQL failed, spawning fix agent...")
    if USE_CLAUDE:
        return fix_sql_claude(bad_sql, error_message)

    # ── Ollama fallback ────────────────────────────────────────────────────────
    schema = get_schema()
    prompt = f"""You are a SQLite SQL expert helping fix NBA stat queries.

Context: These queries are designed to invent custom basketball statistics by computing
a new column using arithmetic formulas on existing player stat columns (like PTS, REB, AST,
STL, BLK, FG3M, FG3A, FG_PCT etc). The computed column represents a brand new stat and
is used in both the SELECT and ORDER BY clauses.

Database schema:
{schema}

The following SQL query has a syntax error:
SQL: {bad_sql}
Error: {error_message}

SQLite alias rules to follow when fixing:
- Aliases CANNOT start with a number (e.g. 3P_Index → use Three_P_Index)
- Aliases with hyphens MUST use backticks (e.g. Two-Way → `Two-Way`)
- The ORDER BY alias must exactly match the SELECT alias
- Stick to letters, numbers, and underscores in aliases

Return ONLY the corrected SQL query, no explanations, no markdown backticks."""

    response = chat(model='gemma3', messages=[{'role': 'user', 'content': prompt}])
    fixed = response.message.content.strip()

    if fixed.startswith("```"):
        fixed = fixed.split("```")[1]
        if fixed.startswith("sql"):
            fixed = fixed[3:]

    return fixed.strip()

In [ ]:
import re as _re

CHARTS_DIR = "charts"
SESSIONS_DIR = "sessions"
os.makedirs(CHARTS_DIR, exist_ok=True)
os.makedirs(SESSIONS_DIR, exist_ok=True)

def _chart_filename(title: str) -> str:
    slug = _re.sub(r'[^\w]+', '_', title.strip().lower()).strip('_') or 'chart'
    return os.path.join(CHARTS_DIR, f"{slug}.png")

def visualize_data(query_id: str, chart_type: str, title: str, subtitle: str,
                   x_col: str, y_col: str, color_col: str = None) -> dict:
    """
    Render a chart from a cached query result. Call run_sql_query first and pass its query_id here.
    Choose the chart type that best fits the data based on the preview and columns returned by run_sql_query:
    - 'bar': comparing a small set of players/teams on one metric (vertical bars, short labels)
    - 'horizontal_bar': leaderboards where player names are labels (names readable on y-axis)
    - 'line': trends over time or ordered sequence (e.g. game-by-game performance)
    - 'scatter': relationship between two numeric variables across many players
    - 'pie': composition/share among a small set (use only if <= 8 slices)
    - 'histogram': distribution of a single numeric column across many rows
    Args:
        query_id: The query_id returned by run_sql_query
        chart_type: One of 'bar', 'horizontal_bar', 'line', 'scatter', 'pie', 'histogram'
        title: Main chart title (e.g. the stat name)
        subtitle: Formula or description shown as a subtitle
        x_col: Column for x-axis / labels (must match a column name from run_sql_query)
        y_col: Column for y-axis / values (must match a column name from run_sql_query)
        color_col: Optional column to color scatter points by (scatter only)
    Returns:
        Dict with status and row count, or error message
    """
    df = _query_cache.get(query_id)
    if df is None:
        return {"error": f"No cached result for query_id '{query_id}'. Run run_sql_query first."}
    try:
        fig, ax = plt.subplots(figsize=(14, 7))
        fig.suptitle(subtitle, fontsize=10, y=0.98, color='gray')
        ax.set_title(title, fontsize=14, fontweight='bold', pad=20)

        if chart_type == 'bar':
            bars = ax.bar(df[x_col].astype(str), df[y_col])
            ax.set_xticklabels(df[x_col].astype(str), rotation=45, ha='right')
            for bar, val in zip(bars, df[y_col]):
                label = f'{val:.1f}' if isinstance(val, float) else str(val)
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                        label, ha='center', va='bottom', fontsize=8)

        elif chart_type == 'horizontal_bar':
            bars = ax.barh(df[x_col].astype(str), df[y_col])
            for bar, val in zip(bars, df[y_col]):
                label = f' {val:.1f}' if isinstance(val, float) else f' {val}'
                ax.text(bar.get_width(), bar.get_y() + bar.get_height() / 2,
                        label, ha='left', va='center', fontsize=8)

        elif chart_type == 'line':
            ax.plot(df[x_col].astype(str), df[y_col], marker='o')
            ax.set_xticklabels(df[x_col].astype(str), rotation=45, ha='right')

        elif chart_type == 'scatter':
            colors = df[color_col] if color_col and color_col in df.columns else None
            sc = ax.scatter(df[x_col], df[y_col], c=colors, alpha=0.7, cmap='viridis')
            if colors is not None:
                plt.colorbar(sc, ax=ax, label=color_col)

        elif chart_type == 'pie':
            ax.pie(df[y_col], labels=df[x_col].astype(str), autopct='%1.1f%%')

        elif chart_type == 'histogram':
            ax.hist(df[y_col], bins=20, edgecolor='black')
            ax.set_xlabel(y_col)
            ax.set_ylabel('Count')

        if chart_type not in ('pie', 'histogram'):
            ax.set_xlabel(x_col)
            ax.set_ylabel(y_col)

        plt.subplots_adjust(bottom=0.25)
        plt.tight_layout()

        filename = _chart_filename(title)
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        print(f"Chart saved: {filename}")
        plt.show()
        return {"status": "displayed", "chart_type": chart_type, "rows": len(df), "file": filename}
    except Exception as e:
        return {"error": str(e)}


In [ ]:
def create_visualization(name: str, description: str, python_code: str,
                         query_id: str, title: str, subtitle: str) -> dict:
    """
    Create and immediately run a custom matplotlib visualization from a cached query result.
    Use this when standard chart types cannot express the stat (e.g. radar charts, bubble plots,
    multi-panel subplots, grouped comparisons, custom annotations).
    Args:
        name: Short snake_case name for the chart function (e.g. 'radar_chart', 'bubble_plot')
        description: What this visualization shows
        python_code: Complete Python function definition. Must define a function `name(df, title, subtitle)`
                     that accepts a pandas DataFrame. Available in scope: pd, plt, math, json, os, np.
        query_id: The query_id returned by run_sql_query
        title: Main chart title
        subtitle: Formula or description shown as subtitle
    Returns:
        Dict with status and row count, or error message
    """
    import numpy as np
    df = _query_cache.get(query_id)
    if df is None:
        return {"error": f"No cached result for query_id '{query_id}'. Run run_sql_query first."}

    safe_ns = {"pd": pd, "plt": plt, "math": math, "json": json, "os": os, "np": np}
    try:
        exec(python_code, safe_ns)
        fn = safe_ns.get(name)
        if fn is None:
            return {"error": f"Function '{name}' not found in provided code"}

        filename = _chart_filename(title)

        original_show = plt.show
        def _saving_show(*args, **kwargs):
            plt.savefig(filename, dpi=150, bbox_inches='tight')
            print(f"Chart saved: {filename}")
            original_show(*args, **kwargs)
        plt.show = _saving_show
        try:
            fn(df, title, subtitle)
            if plt.get_fignums():  # only if a figure is actually open
                plt.show()
        finally:
            plt.show = original_show

        print(f"\n[Custom Viz] {name}: {description}")
        return {"status": "displayed", "viz": name, "rows": len(df), "file": filename}
    except Exception as e:
        return {"error": str(e)}


In [ ]:
import ephem
import sqlite3
import pandas as pd
from datetime import datetime

In [ ]:

DB_PATH = "nba_stats.db"
SQL_MAX_ROWS = 500
PREVIEW_ROWS = 5

# Server-side cache: Claude receives a query_id and never sees the full data.
_query_cache = {}
_query_id_counter = 0


def get_moon_phase(date: str) -> dict:
    """
    Get the moon phase for a given game date.
    Args:
        date: Game date in YYYY-MM-DD format
    Returns:
        Dictionary with moon_pct (0-100) and moon_label (New Moon, Crescent, Quarter, Gibbous, Full Moon)
    """
    try:
        d = ephem.Date(date.replace('-', '/'))
        pct = round(ephem.Moon(d).phase, 1)
        if pct < 10:   label = 'New Moon'
        elif pct < 35: label = 'Crescent'
        elif pct < 65: label = 'Quarter'
        elif pct < 90: label = 'Gibbous'
        else:          label = 'Full Moon'
        return {"moon_pct": pct, "moon_label": label}
    except Exception as e:
        return {"error": str(e)}


def get_player_game_dates(player_name: str) -> dict:
    """
    Get all game dates and basic stats for a specific player this season.
    Args:
        player_name: Full name of the NBA player (e.g. 'LeBron James')
    Returns:
        Dictionary with a list of game dates and basic stats per game
    """
    try:
        conn = sqlite3.connect(DB_PATH)
        df = pd.read_sql("""
            SELECT GAME_DATE, MATCHUP, HOME_AWAY, WL, PTS, REB, AST
            FROM game_logs WHERE PLAYER_NAME = ? ORDER BY GAME_DATE
        """, conn, params=[player_name])
        conn.close()
        return {"games": df.to_dict(orient='records')}
    except Exception as e:
        return {"error": str(e)}


def get_available_tables() -> dict:
    """
    List all tables and views in the NBA database with their column names.
    Returns:
        Dictionary with table/view names and their columns
    """
    try:
        conn = sqlite3.connect(DB_PATH)
        tables = conn.execute(
            "SELECT name, type FROM sqlite_master WHERE type IN ('table','view') ORDER BY type, name"
        ).fetchall()
        result = {}
        for name, kind in tables:
            cols = [row[1] for row in conn.execute(f"PRAGMA table_info({name})").fetchall()]
            result[name] = {"type": kind, "columns": cols}
        conn.close()
        return result
    except Exception as e:
        return {"error": str(e)}


def run_sql_query(sql: str) -> dict:
    """
    Run a SQL query and cache the results server-side. Returns a query_id to pass to
    visualize_data or create_visualization — do NOT try to interpret the full rows yourself.
    Always include LIMIT (max 500 rows enforced automatically).
    Args:
        sql: A valid SQLite SELECT query (include LIMIT to control result size)
    Returns:
        query_id to reference in visualization calls, plus columns/row_count/preview for planning
    """
    global _query_id_counter
    print(f"\nSQL:\n{sql}\n")
    try:
        conn = sqlite3.connect(DB_PATH)
        df = pd.read_sql(sql, conn)
        conn.close()

        total_rows = len(df)
        if total_rows > SQL_MAX_ROWS:
            df = df.head(SQL_MAX_ROWS)
            print(f"[Truncated: {SQL_MAX_ROWS} of {total_rows} rows]")

        _query_id_counter += 1
        query_id = f"q{_query_id_counter}"
        _query_cache[query_id] = df

        return {
            "query_id": query_id,
            "columns": list(df.columns),
            "row_count": len(df),
            "preview": df.head(PREVIEW_ROWS).to_dict(orient='records'),
        }
    except Exception as e:
        return {"error": str(e)}


def get_player_bio(player_name: str) -> dict:
    """
    Get physical and biographical information for a player.
    Args:
        player_name: Full name of the NBA player (e.g. 'Stephen Curry')
    Returns:
        Dictionary with height, weight, college, country, draft info
    """
    try:
        conn = sqlite3.connect(DB_PATH)
        df = pd.read_sql("""
            SELECT PLAYER_NAME, TEAM_ABBREVIATION, HEIGHT, HEIGHT_INCHES,
                   WEIGHT, POSITION, COUNTRY, COLLEGE, DRAFT_YEAR, DRAFT_NUMBER
            FROM player_stats WHERE PLAYER_NAME = ?
        """, conn, params=[player_name])
        conn.close()
        if df.empty:
            return {"error": f"No bio found for {player_name}"}
        return df.iloc[0].to_dict()
    except Exception as e:
        return {"error": str(e)}


In [ ]:
# ── Dynamic Tools ──────────────────────────────────────────────────────────────
DYNAMIC_TOOLS = True  # Toggle: set False to disable AI tool creation
DYNAMIC_TOOLS_FILE = "dynamic_tools.json"
_dynamic_tools = {}  # name → callable, populated by load/create


def load_dynamic_tools():
    global _dynamic_tools
    if not os.path.exists(DYNAMIC_TOOLS_FILE):
        print("No saved dynamic tools found.")
        return
    with open(DYNAMIC_TOOLS_FILE, 'r') as f:
        saved = json.load(f)
    safe_ns = {
        "sqlite3": sqlite3, "pd": pd, "DB_PATH": DB_PATH,
        "ephem": ephem, "json": json, "math": math, "os": os,
    }
    print(f"Loading {len(saved)} saved dynamic tool(s)...")
    for name, entry in saved.items():
        try:
            exec(entry["code"], safe_ns)
            fn = safe_ns[name]
            fn.__doc__ = entry["description"]
            _dynamic_tools[name] = fn
            print(f"  [OK] {name}")
        except Exception as e:
            print(f"  [Fail] {name}: {e}")


def _save_dynamic_tool(name, description, code, reason):
    saved = {}
    if os.path.exists(DYNAMIC_TOOLS_FILE):
        with open(DYNAMIC_TOOLS_FILE, 'r') as f:
            saved = json.load(f)
    saved[name] = {"description": description, "code": code, "reason": reason}
    with open(DYNAMIC_TOOLS_FILE, 'w') as f:
        json.dump(saved, f, indent=2)


In [ ]:
def create_tool(name: str, description: str, python_code: str, reason: str) -> dict:
    """
    Create a new Python tool available for this and future sessions.
    Args:
        name: Function name in snake_case (no spaces or hyphens)
        description: What this tool does (used as its docstring)
        python_code: Complete Python function definition as a string
        reason: Why this tool is being created
    Returns:
        Dictionary with status and tool name
    """
    print(f"\n[New Tool] {name}")
    print(f"  Why:  {reason}")
    print(f"  Does: {description}")

    safe_ns = {
        "sqlite3": sqlite3, "pd": pd, "DB_PATH": DB_PATH,
        "ephem": ephem, "json": json, "math": math, "os": os,
    }
    try:
        exec(python_code, safe_ns)
        fn = safe_ns.get(name)
        if fn is None:
            return {"error": f"Function '{name}' not found in provided code"}
        fn.__doc__ = description
        _dynamic_tools[name] = fn
        _save_dynamic_tool(name, description, python_code, reason)
        print(f"  [OK] Registered and saved to {DYNAMIC_TOOLS_FILE}")
        return {"status": "registered", "tool": name}
    except Exception as e:
        print(f"  [Error] {e}")
        return {"error": str(e)}


load_dynamic_tools()

In [ ]:
def assemble_tools() -> list:
    base = [
        get_moon_phase,
        get_player_game_dates,
        get_available_tables,
        run_sql_query,
        get_player_bio,
        visualize_data,
        create_visualization,
    ]
    if DYNAMIC_TOOLS:
        base += list(_dynamic_tools.values())
        base.append(create_tool)
    return base

In [ ]:
def run_agentic_loop(messages):
    """Run the model in a tool-call loop until it returns a final text answer.

    Dispatches to run_agentic_loop_claude (Anthropic API) or the Ollama loop
    depending on the USE_CLAUDE toggle set at the top of the notebook.
    """
    if USE_CLAUDE:
        return run_agentic_loop_claude(messages)

    # ── Ollama fallback ────────────────────────────────────────────────────────
    while True:
        tools = assemble_tools()
        tool_registry = {fn.__name__: fn for fn in tools}

        response = chat(
            model=model,
            messages=messages,
            tools=tools,
            options={"temperature": 0.1}
        )

        if not response.message.tool_calls:
            return response.message.content or ""

        messages.append(response.message)

        for tc in response.message.tool_calls:
            fn_name = tc.function.name
            fn_args = tc.function.arguments or {}
            fn = tool_registry.get(fn_name)
            if fn is None:
                result = {"error": f"Unknown tool: {fn_name}"}
            else:
                try:
                    result = fn(**fn_args)
                except Exception as e:
                    result = {"error": str(e)}

            messages.append({
                'role': 'tool',
                'content': json.dumps(result),
            })

In [ ]:
def invent_stat_and_query(user_question):
    if USE_CLAUDE:
        preamble = ''
    else:
        schema = get_schema()
        preamble = (
            f"You are an NBA statistician with access to tools and this SQLite database schema:\n\n"
            f"{schema}\n\n"
        )

    prompt = f"""{preamble}The user wants to know: "{user_question}"

You are the ORCHESTRATOR. Your job is to plan and dispatch — not to read or interpret raw data.
The pipeline is: design formula → write SQL → run query (get query_id) → visualize using query_id → summarize.
Query results are cached server-side. You receive only column names, row count, and a 5-row preview.
Do NOT wait to see all the data before deciding how to visualize — use the preview and column names.

Complete ALL four steps:

STEP 1 — Design the formula and query plan.
- What concept is the user asking about? What makes a player elite at this?
- Does answering require data NOT in the database (elevation, travel distance, etc.)?
  If yes, call create_tool FIRST to register a hardcoded lookup, then use it alongside the query.
- Design a formula using real basketball reasoning and available columns (PTS, REB, AST, STL,
  BLK, FG_PCT, FG3M, FG3A, TOV, MIN, GP, HEIGHT_INCHES, WEIGHT, etc.).
- Name the stat clearly (no leading digit, underscores not hyphens).
- Decide: per-game or per-minute? Minimum GP filter? LIMIT for the leaderboard?

STEP 2 — Query the database.
- Call run_sql_query with the formula baked into the SELECT.
- You will receive back: query_id, columns, row_count, and a 5-row preview.
- Use the preview to confirm column names and value ranges — do not try to read the full results.
- SQLite alias rules: alias cannot start with a digit; use underscores not hyphens.

STEP 3 — Visualize using the query_id.
- Pass the query_id (not records) to visualize_data or create_visualization.
- Pick chart type from the preview and column names alone:
    'horizontal_bar' — leaderboards / rankings
    'bar'            — small side-by-side comparisons (≤10 items, short labels)
    'scatter'        — relationship between two numeric variables
    'line'           — trend over time or ordered sequence
    'pie'            — share/composition for a small group (≤8 slices only)
    'histogram'      — distribution across many players
- For multi-dimensional data, subplots, radar/spider, bubble, or grouped charts:
  use create_visualization and write a function that accepts (df, title, subtitle).

STEP 4 — Return a short plain-text summary: stat name, formula, and key finding from the preview."""

    messages = [{'role': 'user', 'content': prompt}]
    return run_agentic_loop(messages)


In [ ]:
def analyze(question):
    print(f"\nQuestion: {question}")
    print("Analyzing...")
    summary = invent_stat_and_query(question)
    if summary:
        print(f"\n{summary}")

In [ ]:
import tkinter as tk
from tkinter import ttk, scrolledtext, filedialog
import threading
import sys
from datetime import datetime


class _GUIStream:
    """Redirects sys.stdout to the GUI history box (thread-safe via root.after)."""
    def __init__(self, app):
        self._app = app

    def write(self, text):
        if text:
            self._app.root.after(0, self._app._append, text)

    def flush(self):
        pass


class NBAStatsApp:
    def __init__(self, root):
        self.root = root
        self.root.title("NBA Stats AI")
        self.root.minsize(700, 500)
        self.output_path = os.path.join(
            SESSIONS_DIR, f"nba_session_{datetime.now().strftime('%Y-%m-%d')}.txt"
        )
        self._active = False
        self._stream = _GUIStream(self)
        self._build_ui()
        self._append(
            f"Session started. Logs → {self.output_path} | Charts → {CHARTS_DIR}/\n"
            "Ask any NBA stats question and press Enter or click Ask.\n"
            + "─" * 60 + "\n"
        )

    def _build_ui(self):
        self.root.columnconfigure(0, weight=1)
        self.root.rowconfigure(0, weight=1)

        history_frame = ttk.Frame(self.root)
        history_frame.grid(row=0, column=0, sticky='nsew', padx=8, pady=(8, 0))
        history_frame.columnconfigure(0, weight=1)
        history_frame.rowconfigure(0, weight=1)

        self.history = scrolledtext.ScrolledText(
            history_frame, wrap=tk.WORD, state='disabled',
            font=('Consolas', 10), relief='flat'
        )
        self.history.grid(row=0, column=0, sticky='nsew')

        input_frame = ttk.Frame(self.root)
        input_frame.grid(row=1, column=0, sticky='ew', padx=8, pady=4)
        input_frame.columnconfigure(0, weight=1)

        self.question_var = tk.StringVar()
        self.entry = ttk.Entry(input_frame, textvariable=self.question_var, font=('Consolas', 11))
        self.entry.grid(row=0, column=0, sticky='ew', padx=(0, 6))
        self.entry.bind('<Return>', lambda _: self._submit())

        self.ask_btn = ttk.Button(input_frame, text='Ask', command=self._submit, width=8)
        self.ask_btn.grid(row=0, column=1)

        ctrl_frame = ttk.Frame(self.root)
        ctrl_frame.grid(row=2, column=0, sticky='ew', padx=8, pady=(0, 4))
        ttk.Button(ctrl_frame, text='Change log file…', command=self._pick_file).pack(side='left', padx=(0, 6))
        ttk.Button(ctrl_frame, text='Clear history', command=self._clear).pack(side='left')
        self.file_label = ttk.Label(ctrl_frame, text=f"→ {self.output_path}", foreground='gray')
        self.file_label.pack(side='left', padx=10)

        self.status_var = tk.StringVar(value='Ready')
        ttk.Label(self.root, textvariable=self.status_var, relief='sunken', anchor='w', padding=(4, 2)
                  ).grid(row=3, column=0, sticky='ew')
        self.entry.focus()

    def _submit(self):
        question = self.question_var.get().strip()
        if not question or self._active:
            return
        self.question_var.set('')
        self._active = True
        self.ask_btn.state(['disabled'])
        self.status_var.set('Analyzing…')
        self._append(f"\n[{datetime.now().strftime('%H:%M:%S')}] Q: {question}\n")
        threading.Thread(target=self._run, args=(question,), daemon=True).start()

    def _run(self, question):
        old_stdout = sys.stdout
        sys.stdout = self._stream
        try:
            result = invent_stat_and_query(question)
        except Exception as e:
            result = f"[ERROR] {e}"
        finally:
            sys.stdout = old_stdout
        self.root.after(0, self._on_done, question, result)

    def _on_done(self, question, result):
        ts = datetime.now().strftime('%H:%M:%S')
        self._append(f"\n[{ts}] Summary: {result}\n" + "─" * 60 + "\n")
        self._save(question, result)
        self.status_var.set('Ready')
        self.ask_btn.state(['!disabled'])
        self._active = False

    def _pick_file(self):
        path = filedialog.asksaveasfilename(
            defaultextension='.txt',
            filetypes=[('Text files', '*.txt'), ('All files', '*.*')],
            initialfile=self.output_path,
        )
        if path:
            self.output_path = path
            self.file_label.config(text=f"→ {path}")

    def _clear(self):
        self.history.config(state='normal')
        self.history.delete('1.0', tk.END)
        self.history.config(state='disabled')

    def _append(self, text):
        self.history.config(state='normal')
        self.history.insert(tk.END, text)
        self.history.see(tk.END)
        self.history.config(state='disabled')

    def _save(self, question, result):
        entry = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}]\nQ: {question}\nA: {result}\n{'─'*60}\n\n"
        try:
            with open(self.output_path, 'a', encoding='utf-8') as f:
                f.write(entry)
        except Exception as e:
            self._append(f"[WARN] Could not save: {e}\n")





In [ ]:
root = tk.Tk()
root.geometry('900x650')
NBAStatsApp(root)
root.mainloop()